# 06_mllm / mllm_inference.ipynb

**Purpose:** Generate **free-form natural-language explanations** for a stratified sample of 500 GSAF predictions using Gemini 2.5 Flash.

**Why these explanations matter:**
- The GSAF model is a black-box gating network. Reviewers will ask *why* the model flagged a pair as out-of-context.
- These explanations are not used for training; they are post-hoc rationales that ground the model's verdict in observable facts about the image and caption.
- The next notebook (`chain_of_thought.ipynb`) compares this free-form approach against a structured chain-of-thought version.

**Sample design — 500 pairs total:**
- ~150 from each test set (NewsCLIPpings, MMFakeBench, VERITE) where local images are available
- Within each dataset, balanced across the 4 confusion-matrix buckets (TP / FP / FN / TN)
- Preference for high-confidence predictions (clearer signal for the LLM)

**Inputs:**
- `cache/fusion/fusion_predictions.csv`
- `data/processed/ooc_pairs/manifests/all.csv`
- `paper/tables/qualitative_examples.csv` (we already chose 24 illustrative pairs there — those are auto-included)
- Gemini API key in `.env` (set up in `gemini_setup.ipynb`)

**Outputs:**
- `cache/mllm/explanations_freeform.csv` — pair_id, dataset, bucket, gates, explanation text
- `cache/mllm/explanations_freeform.jsonl` — one JSON line per pair (raw)
- `paper/tables/mllm_explanations_sample.csv` — top 20 picks for the paper appendix

**Free-tier rate-limit note:**
At 10 RPM and 250 RPD, 500 calls take ~2 calendar days. The notebook is restartable — already-cached responses are skipped.


In [ ]:
# Imports & path setup
from __future__ import annotations
import json, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

NB_DIR = Path.cwd()
ROOT = NB_DIR
for parent in [NB_DIR] + list(NB_DIR.parents):
    if (parent / "src" / "utils" / "paths.py").exists():
        ROOT = parent
        break
sys.path.insert(0, str(ROOT))

from src.utils.paths import (
    ALL_MANIFEST, FUSION_CACHE, PROCESSED_IMAGES, VERITE_IMAGES,
    PAPER, TABLES, FIGURES,
)

CACHE_MLLM = ROOT / "cache" / "mllm"
CACHE_MLLM.mkdir(parents=True, exist_ok=True)

# Verify gemini_setup.ipynb has been run
sentinel = CACHE_MLLM / ".gemini_setup_ok"
if not sentinel.exists():
    raise RuntimeError(f"Run 06_mllm/gemini_setup.ipynb first — missing sentinel at {sentinel}")
print("Setup sentinel found:", json.loads(sentinel.read_text()))


## 1. Load fusion predictions + manifest, compute buckets

In [ ]:
thresholds_path = FUSION_CACHE / "fusion_thresholds.json"
fusion_path = FUSION_CACHE / "fusion_predictions.csv"
preds = pd.read_csv(fusion_path)
manifest = pd.read_csv(ALL_MANIFEST)
thresholds = json.loads(thresholds_path.read_text()) if thresholds_path.exists() else {}

# Identify columns
def find_col(df, cands):
    for c in cands:
        if c in df.columns: return c
    return None

prob_col = find_col(preds, ["score", "fusion_prob", "prob_fusion", "y_prob_fusion", "y_prob"])
y_col    = find_col(preds, ["label", "y_true", "target"])
pid_col  = find_col(preds, ["pair_id", "id"])
es_col   = find_col(preds, ["eval_split"])
ga, gb, gc = (find_col(preds, [c]) for c in ["gate_a", "gate_b", "gate_c"])

# Threshold per row
def thresh_for(es):
    return float(thresholds.get(es, thresholds.get("default", 0.5)))
preds["_threshold"] = preds[es_col].map(thresh_for)
preds["_y_pred"] = (preds[prob_col].astype(float) >= preds["_threshold"]).astype(int)
preds["_bucket"] = np.where(
    (preds[y_col] == 1) & (preds["_y_pred"] == 1), "TP",
    np.where((preds[y_col] == 0) & (preds["_y_pred"] == 1), "FP",
    np.where((preds[y_col] == 1) & (preds["_y_pred"] == 0), "FN", "TN")))

print("Predictions shape:", preds.shape)
print("Bucket counts (test splits only):")
test_only = preds[preds[es_col].str.contains("test", na=False)]
print(test_only.groupby([find_col(preds, ["dataset"]), "_bucket"]).size())


## 2. Resolve image paths and merge captions

In [ ]:
ds_col = find_col(preds, ["dataset"])
sp_col = find_col(preds, ["split"])

merged = preds.merge(
    manifest[["pair_id", "caption", "image_path"]].drop_duplicates("pair_id"),
    on="pair_id", how="left"
)

def resolve_image(p):
    if pd.isna(p) or not p: return None
    pp = Path(str(p))
    if pp.is_absolute() and pp.exists():
        return pp
    for base in [PROCESSED_IMAGES, VERITE_IMAGES, ROOT / "data" / "raw" / "mmfakebench", ROOT]:
        cand = base / pp.name
        if cand.exists(): return cand
        cand2 = base / pp
        if cand2.exists(): return cand2
    return None

merged["_image_path"] = merged["image_path"].map(resolve_image)
print("Rows with local image:", merged["_image_path"].notna().sum(), "/", len(merged))
print("By dataset:")
print(merged.groupby(ds_col)["_image_path"].apply(lambda s: s.notna().sum()))


## 3. Stratified sampling — 500 pairs

In [ ]:
TOTAL_BUDGET = 500
PER_DATASET = TOTAL_BUDGET // 3   # 166 each, rounded; one dataset gets the +2

# Restrict to test splits with local images
pool = merged[
    merged[es_col].str.contains("test", na=False) &
    merged["_image_path"].notna() &
    merged["caption"].notna()
].copy()
print("Pool size (test rows with image+caption):", len(pool))
print("By dataset:", pool[ds_col].value_counts().to_dict())

BUCKETS = ["TP", "FP", "FN", "TN"]

selected = []
rng = np.random.default_rng(42)

for ds_name in pool[ds_col].unique():
    ds_pool = pool[pool[ds_col] == ds_name]
    quota = PER_DATASET
    per_bucket = max(1, quota // len(BUCKETS))
    for bk in BUCKETS:
        sub = ds_pool[ds_pool["_bucket"] == bk].copy()
        if sub.empty:
            continue
        # Score: prefer high-confidence picks (interesting either way)
        if bk in ("TP", "FP"):
            sub["_score"] = sub[prob_col].astype(float)            # confidently positive
        else:
            sub["_score"] = 1.0 - sub[prob_col].astype(float)      # confidently negative
        sub = sub.sort_values("_score", ascending=False).head(per_bucket * 3)
        # Random tiebreak among top candidates
        idx = rng.choice(sub.index, size=min(per_bucket, len(sub)), replace=False)
        selected.append(sub.loc[idx])

sample_df = pd.concat(selected, ignore_index=True)
print(f"\nSelected {len(sample_df)} pairs:")
print(sample_df.groupby([ds_col, "_bucket"]).size())

# Drop columns we don't need downstream
keep = [pid_col, ds_col, sp_col, y_col, "_y_pred", "_bucket", prob_col, ga, gb, gc,
        "caption", "_image_path"]
keep = [c for c in keep if c is not None]
sample_df = sample_df[keep].reset_index(drop=True)
sample_df.head()


## 4. Free-form prompt template

In [ ]:
FREEFORM_PROMPT = '''You are a fact-checking analyst. You are given an image and a caption that was published with it.

The caption is:
"""{caption}"""

Your task: assess whether the image plausibly matches the caption.

Write a short response with three labelled lines:

IMAGE: One sentence describing what is visible in the image (objects, setting, people, mood).
CAPTION_CHECK: One sentence stating whether the image plausibly supports the caption, or whether something seems off (mismatched location, wrong people, time-of-day clue, anachronism, etc.).
VERDICT: Exactly one word — either MATCH or MISMATCH.

Be specific. Do not say "I cannot tell" unless the image is genuinely unreadable.'''

print(FREEFORM_PROMPT.format(caption="<EXAMPLE CAPTION>")[:500])


## 5. Generate explanations (with cache + rate-limit)

In [ ]:
from src.utils.gemini_client import GeminiClient

client = GeminiClient(rpm_limit=9)   # leave 1 request headroom under 10 RPM
print("Using model:", client.model_name)

OUT_CSV   = CACHE_MLLM / "explanations_freeform.csv"
OUT_JSONL = CACHE_MLLM / "explanations_freeform.jsonl"

# If a partial run exists, skip already-processed pair_ids
done_ids = set()
if OUT_CSV.exists():
    prev = pd.read_csv(OUT_CSV)
    done_ids = set(prev[pid_col].astype(str).tolist())
    print(f"Found {len(done_ids)} already-processed pairs in {OUT_CSV.name}")

records = []
errors = []

for i, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Gemini calls"):
    pid = str(row[pid_col])
    if pid in done_ids:
        continue
    cap = str(row["caption"])
    img = row["_image_path"]
    prompt = FREEFORM_PROMPT.format(caption=cap[:600])
    try:
        text = client.generate(prompt=prompt, image_path=img,
                               use_cache=True, temperature=0.2, max_output_tokens=300)
        rec = {
            "pair_id":  pid,
            "dataset":  row[ds_col],
            "split":    row[sp_col],
            "y_true":   int(row[y_col]),
            "y_pred":   int(row["_y_pred"]),
            "bucket":   row["_bucket"],
            "fusion_prob": float(row[prob_col]),
            "gate_a":   float(row[ga]) if ga in row else None,
            "gate_b":   float(row[gb]) if gb in row else None,
            "gate_c":   float(row[gc]) if gc in row else None,
            "caption":  cap,
            "explanation": text,
        }
        records.append(rec)
        # Append-write every record so we don't lose work on a crash
        with open(OUT_JSONL, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    except Exception as e:
        errors.append({"pair_id": pid, "error": str(e)})
        print(f"  [error] {pid}: {e}")
        # Persist partial progress
        if records:
            df_partial = pd.DataFrame(records)
            df_partial.to_csv(OUT_CSV, index=False, mode="w")
        continue

# Combine with previous run, if any
if OUT_CSV.exists():
    prev = pd.read_csv(OUT_CSV)
    new_df = pd.DataFrame(records)
    full = pd.concat([prev, new_df], ignore_index=True).drop_duplicates(pid_col, keep="last")
else:
    full = pd.DataFrame(records)
full.to_csv(OUT_CSV, index=False)
print(f"\nDone. Total explanations on disk: {len(full)}")
print(f"Errors this run: {len(errors)}")
if errors:
    print(pd.DataFrame(errors).head())


## 6. Parse VERDICT from explanations and compare with model

In [ ]:
import re

def parse_verdict(text):
    if not isinstance(text, str): return None
    m = re.search(r"VERDICT\s*:\s*(MATCH|MISMATCH)", text, re.IGNORECASE)
    if m:
        return m.group(1).upper()
    # Fallback: last non-empty line
    for line in reversed(text.strip().splitlines()):
        line = line.strip().upper()
        if "MISMATCH" in line: return "MISMATCH"
        if "MATCH" in line: return "MATCH"
    return None

full = pd.read_csv(OUT_CSV)
full["mllm_verdict"] = full["explanation"].map(parse_verdict)
full["mllm_pred_ooc"] = full["mllm_verdict"].map({"MISMATCH": 1, "MATCH": 0})

print("Verdict counts:")
print(full["mllm_verdict"].value_counts(dropna=False))

# Agreement with the GSAF model
agree = (full["mllm_pred_ooc"] == full["y_pred"]).mean()
print(f"\nMLLM ↔ GSAF agreement: {agree:.3f}  (n={full['mllm_pred_ooc'].notna().sum()})")

# Agreement with ground truth
mask = full["mllm_pred_ooc"].notna()
mllm_acc = (full.loc[mask, "mllm_pred_ooc"] == full.loc[mask, "y_true"]).mean()
gsaf_acc = (full.loc[mask, "y_pred"]        == full.loc[mask, "y_true"]).mean()
print(f"\nOn pairs MLLM produced a verdict for (n={mask.sum()}):")
print(f"  MLLM zero-shot accuracy: {mllm_acc:.3f}")
print(f"  GSAF accuracy on same:   {gsaf_acc:.3f}")

full.to_csv(OUT_CSV, index=False)


## 7. Pick best 20 examples for paper appendix

In [ ]:
# Prefer: clean verdict parsed, balanced across datasets, mix of correct + interesting failures
def quality(row):
    if pd.isna(row.get("mllm_verdict")): return -1
    score = 0
    # length (prefer well-formed responses)
    text = str(row.get("explanation", ""))
    if 100 < len(text) < 800: score += 2
    if "IMAGE:" in text and "CAPTION_CHECK:" in text: score += 3
    # interesting buckets
    if row["bucket"] in ("FP", "FN", "TP"): score += 1
    # confidence
    score += abs(float(row["fusion_prob"]) - 0.5) * 2
    return score

full["_quality"] = full.apply(quality, axis=1)
top = (full[full["_quality"] > 0]
       .sort_values("_quality", ascending=False)
       .groupby("dataset", group_keys=False)
       .head(7)
       .head(20))

paper_cols = ["pair_id", "dataset", "bucket", "y_true", "y_pred", "mllm_verdict",
              "fusion_prob", "caption", "explanation"]
top[paper_cols].to_csv(TABLES / "mllm_explanations_sample.csv", index=False)
print("Saved:", TABLES / "mllm_explanations_sample.csv")
top[paper_cols].head(10)


---
**Done.** Free-form explanations saved. Next notebook: `chain_of_thought.ipynb` runs the same 500 pairs through a structured CoT prompt and compares verdict accuracy.